# Lab 08-04 — Local vs global search over the GraphRAG index (HotpotQA)

**Track 08 · GraphRAG** — how the *same* graph index answers two very different ways.

An entity graph over a corpus can be queried two ways, and the choice is a real engineering tradeoff. **Local search** starts from the question's own entities, links them into the graph, expands one hop to their neighbors, and answers from the passages those entities live in — precise, but blind to any answer that lives in a *distant* part of the corpus. **Global search** ignores the graph topology for retrieval: it embeds the question, ranks the precomputed community summaries by similarity, and answers from the top ones — it sees the whole corpus (compressed into summaries), but the answer must survive the map/reduce compression. This lab builds a per-question index over **HotpotQA** dev questions and scores both strategies against the dataset's gold answer and gold paragraphs (`supporting_facts`).

```text
hotpotqa dev questions (3, ~16 local LLM calls each)
  -> entity extractor: OllamaLLM qwen2.5-coder:7b (localhost:11434)
  -> local BGE embeddings on CPU (entity linking + global query embedding)
  -> tools/graph build_entity_graph over each question's 10 paragraphs
  -> detect communities + community summaries (tools/graphrag)
  -> local_search vs global_search (tools/graphrag)
  -> score 0/1 flags against gold answer + gold paragraphs
  -> verification gate (--verify)
```

The graph primitives live in `tools/graph.py` and `tools/graphrag.py` — the shared blocks this whole track builds on.


## Setup

This notebook mirrors `curriculum/08-graphrag/04-local-global.py` exactly — the same verified code, split into cells. You can run it from anywhere: the imports cell walks up to the repo root and cd's into it, so every `Data/...` path resolves just like the lab script.

Two local prerequisites must hold — there are **no API calls** in this lab, so it costs zero quota:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM that extracts entities, summarizes communities, and writes both answers (`llms/ollama.py` talks to it through `langchain-ollama`). If the server is not up, every LLM call fails and the experiment never runs.
- **Local BGE embeddings** through `sentence-transformers` — `BAAI/bge-base-en-v1.5` on CPU (`embeddings/bge.py`), used for entity linking in local search and for embedding the question in global search. Everything local, nothing leaves the machine.

From the terminal, the lab runs as:

```bash
python curriculum/08-graphrag/04-local-global.py          # run + demo
python curriculum/08-graphrag/04-local-global.py --verify # verification gate
```

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   langchain-ollama       -> OllamaLLM over the local qwen2.5-coder:7b model (llms/ollama.py)
#   sentence-transformers  -> local BGE embeddings (embeddings/bge.py)
#   networkx               -> the entity graph and its community detection (tools/graph.py, tools/graphrag.py)
%pip install langchain-ollama sentence-transformers networkx


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from embeddings.bge import BGEEmbedding  # noqa: E402
from llms.ollama import OllamaLLM  # noqa: E402
from tools.graph import build_entity_graph  # noqa: E402
from tools.graphrag import (  # noqa: E402
    community_summaries,
    detect_communities,
    global_search,
    local_search,
)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_QUESTIONS = 3` because each question costs roughly **16 local LLM calls** — entity extraction, community summaries, and both searches — so three questions show the local/global tradeoff clearly without a long wait. `MAX_COMMUNITIES = 4` caps how many community summaries each question builds. `LINK_TOP_N = 6` and `LINK_THRESHOLD = 0.55` govern local search: how aggressively the question's entities are linked into the graph and how many passages the search may surface. `BGE_DEVICE = "cpu"` keeps embeddings off the GPU because Ollama already holds most of the VRAM.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
HOTPOT_PATH = Path("Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json")
N_QUESTIONS = 3  # each question costs ~16 LLM calls; keep the lab fast
MAX_COMMUNITIES = 4  # per-question cap on community summaries
LINK_TOP_N = 6  # max passages local search may surface
LINK_THRESHOLD = 0.55  # minimum cosine for entity linking
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load — first N HotpotQA questions (each has 10 context paragraphs)

HotpotQA's dev set is one JSON file of records, and every record is a self-contained micro-dataset: the question, the answer, **10 candidate paragraphs** in `context`, and `supporting_facts` — the paragraph titles HotpotQA marks as required evidence. That layout lets each question run its own GraphRAG experiment: build an index over the 10 paragraphs, run both searches, and score against the record's own ground truth. No external qrels file is needed — the gold paragraphs travel with the question.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N HotpotQA questions (each has 10 context paragraphs)
# --------------------------------------------------------------------------
def load_questions(path: Path, n: int) -> list[dict]:
    """Return the first ``n`` questions from the dev set."""
    with open(path) as f:
        records = json.load(f)
    return records[:n]


def question_passages(rec: dict) -> tuple[list[str], list[str]]:
    """Return (passage_texts, paragraph_titles) for the 10 context paragraphs."""
    texts: list[str] = []
    titles: list[str] = []
    for title, sentences in rec["context"]:
        titles.append(title)
        texts.append(" ".join(sentences))
    return texts, titles


def gold_paragraphs(rec: dict) -> set[str]:
    """The paragraph titles HotpotQA marks as required evidence."""
    return {title for title, _ in rec["supporting_facts"]}


def answer_contains(gold: str, answer: str) -> bool:
    """Normalized substring check: is the gold answer inside the answer?"""
    return gold.strip().lower() in answer.strip().lower()


## 3. Experiment — build a per-question index, run both searches, evaluate

The two strategies retrieve by different routes, and that difference is the point of the lab:

- **Local search** links the question's entities into the graph, expands one hop to their neighbors, and answers from the passages those entities live in. Precise — it only sees passages that touch the question — but it fails when the question's entities are not in the graph or the answer needs a *distant* part of the corpus.
- **Global search** ignores graph topology for retrieval: it embeds the question, ranks the precomputed community summaries by similarity, and answers from the top ones. It sees the whole corpus compressed into summaries, so it can handle corpus-level questions, but the answer must survive the map/reduce compression.

Both searches reuse the per-question graph from the earlier labs. Scores are 0/1 flags: whether the gold answer appears inside each strategy's answer (`answer_contains`) and whether local search surfaced at least one gold paragraph (`local_gold_para_recall`).


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — build a per-question index, run both searches, evaluate
# --------------------------------------------------------------------------
def run_one_question(rec: dict, llm, embedder) -> dict:
    passages, titles = question_passages(rec)
    gold = rec["answer"]
    gold_titles = gold_paragraphs(rec)

    graph = build_entity_graph(passages, llm)
    communities = detect_communities(graph, seed=42)
    summaries = community_summaries(
        llm, graph, communities, max_communities=MAX_COMMUNITIES
    )

    local = local_search(
        rec["question"], llm, embedder, graph, passages,
        top_n=LINK_TOP_N, threshold=LINK_THRESHOLD,
    )
    glob = global_search(rec["question"], llm, embedder, summaries)

    local_gold_ids = [
        titles[i] for i in local["retrieved_ids"]
        if titles[i] in gold_titles
    ]
    return {
        "question": rec["question"],
        "gold": gold,
        "gold_titles": sorted(gold_titles),
        "local": {
            "answer": local["answer"],
            "linked": local["linked"],
            "gold_titles_retrieved": local_gold_ids,
        },
        "global": {
            "answer": glob["answer"],
        },
        "scores": {
            "local_answer_contains_gold": answer_contains(gold, local["answer"]),
            "global_answer_contains_gold": answer_contains(gold, glob["answer"]),
            "local_gold_para_recall": len(local_gold_ids) > 0,
        },
    }


def run_experiment() -> dict:
    questions = load_questions(HOTPOT_PATH, N_QUESTIONS)
    llm = OllamaLLM()
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device=BGE_DEVICE)

    t0 = time.perf_counter()
    rows = [run_one_question(rec, llm, embedder) for rec in questions]
    total_s = time.perf_counter() - t0

    return {
        "rows": rows,
        "total_s": total_s,
        "agg": {
            "local_answer_contains_gold": sum(
                r["scores"]["local_answer_contains_gold"] for r in rows
            ),
            "global_answer_contains_gold": sum(
                r["scores"]["global_answer_contains_gold"] for r in rows
            ),
            "local_gold_para_recall": sum(
                r["scores"]["local_gold_para_recall"] for r in rows
            ),
            "questions": len(rows),
        },
    }


## 4. Demo

The demo prints one block per question — the gold answer, which entities local search linked, and the 0/1 hits for both strategies — then aggregates the flags and a takeaway. The interesting signal is the *disagreement* between the two rows: when the answer needs a bridge entity that is in the graph, local search tends to win; when the question is corpus-level and only the compressed summaries carry the answer, global search holds up better.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 08-04 — Local vs global search over the GraphRAG index")
    print(f"{exp['agg']['questions']} questions in {exp['total_s']:.1f}s")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        print(f"\nQ{i}: {row['question'][:90]}")
        print(f"    gold answer : {row['gold']}")
        print(f"    gold paras  : {', '.join(row['gold_titles'])[:80]}")
        print(f"    linked      : {row['local']['linked']}")
        print(f"    local  hit  : {row['scores']['local_answer_contains_gold']} "
              f"(gold paras {row['local']['gold_titles_retrieved']})")
        print(f"    global hit  : {row['scores']['global_answer_contains_gold']}")
        print(f"    local  ans  : {row['local']['answer'][:110]}")
        print(f"    global ans  : {row['global']['answer'][:110]}")

    a = exp["agg"]
    print(f"\n[5] Aggregates over {a['questions']} questions")
    print(f"    local  answer contains gold : {a['local_answer_contains_gold']}/{a['questions']}")
    print(f"    global answer contains gold : {a['global_answer_contains_gold']}/{a['questions']}")
    print(f"    local  surfaced a gold para : {a['local_gold_para_recall']}/{a['questions']}")

    print(f"\n[6] Takeaway")
    print("    Local search is entity-precise but narrow: it only sees")
    print("    passages that touch the question's entities, so it shines on")
    print("    multi-hop questions whose bridge entities are in the graph.")
    print("    Global search trades that precision for recall: the question")
    print("    is answered from compressed community summaries, so it can")
    print("    generalize but may lose exact facts during map/reduce.")


## 5. Verification gate

The lab ships a `--verify` gate: hard checks the experiment must clear — exactly `N_QUESTIONS` questions processed, local search actually linked entities on at least one question, every answer non-empty, at least one gold paragraph surfaced, and all per-question scores reportable 0/1 flags. The gate turns "the lab ran" into "the lab ran *correctly*" — the same discipline every lab in this track applies.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    a = exp["agg"]

    checks.append((f"exactly {N_QUESTIONS} questions processed",
                   a["questions"] == N_QUESTIONS))
    checks.append(("local search linked entities for >= 1 question",
                   any(r["local"]["linked"] for r in exp["rows"])))
    checks.append(("every answer is non-empty",
                   all(r["local"]["answer"].strip()
                       for r in exp["rows"])
                   and all(r["global"]["answer"].strip()
                           for r in exp["rows"])))
    checks.append(("local search surfaced a gold paragraph for >= 1 question",
                   a["local_gold_para_recall"] >= 1))
    checks.append(("per-question scores are 0/1 flags (reportable)",
                   all(v in (0, 1)
                       for r in exp["rows"]
                       for v in r["scores"].values())))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Three questions × roughly 16 local LLM calls each — extraction, community summaries, and both searches — lands around **6–7 minutes** on a local Ollama model. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces.


In [ ]:
verify_gate(exp)
